In [2]:
#!pip install transformers seqeval evaluate datasets

### Dataset

In [3]:
from datasets import load_dataset

raw_datasets = load_dataset("lfcc/portuguese_ner")
print(raw_datasets)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/266k [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/67.4k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/3716 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/930 [00:00<?, ? examples/s]

DatasetDict({
    train: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 3716
    })
    test: Dataset({
        features: ['tokens', 'ner_tags'],
        num_rows: 930
    })
})


In [4]:
label_list = raw_datasets["train"].features["ner_tags"].feature.names
print(label_list)

['O', 'B-Data', 'I-Data', 'B-Local', 'I-Local', 'B-Organizacao', 'I-Organizacao', 'B-Pessoa', 'I-Pessoa', 'B-Profissao', 'I-Profissao']


In [5]:
raw_datasets["train"]["tokens"]

Column([['Filiação', ':', 'Antonio', 'Joaquim', 'Aguiar', 'e', 'Engracia', 'Maria', '.', 'Natural', 'e/ou', 'residente', 'em', 'CUNHA', ',', 'Santa', 'Maria', ',', 'actual', 'concelho', 'de', 'PAREDES', 'COURA', 'e', 'distrito', '(', 'ou', 'país', ')', 'Viana', 'do', 'Castelo', '.'], ['Filiação', ':', 'Domingos', 'Pires', 'e', 'Comba', 'Fernandes', '.', 'Natural', 'e/ou', 'residente', 'em', 'VALONGO', 'MILHAIS', ',', 'Sao', 'Goncalo', ',', 'actual', 'concelho', 'de', 'MURCA', 'e', 'distrito', '(', 'ou', 'país', ')', 'VILA', 'REAL', '.'], ['Termo', 'de', 'justificação', 'do', 'baptismo', 'de', 'Pedro', 'Gonçalves', 'Coques', ',', 'nascido', 'em', '29.06.1876', 'e', 'baptizado', '"', '(', '…', ')', 'por', 'dias', 'do', 'mês', 'de', 'Julho', 'do', 'dito', 'ano', ',', '(', '…', ')', '"', ',', 'na', 'igreja', 'do', 'Jardim', 'do', 'Mar', ',', 'Calheta', '.'], ['Doc.danificado', '.'], ['1898-11-01', '/', '1898-11-01']])

### Pre-Processing

In [6]:

checkpoint = "neuralmind/bert-base-portuguese-cased"
from transformers import AutoTokenizer
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

config.json:   0%|          | 0.00/647 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/43.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/2.00 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

In [7]:
tokenized_inputs = tokenizer("A aula de SPLN é muito interessante!")
print(tokenized_inputs)

#print(tokenizer.convert_ids_to_tokens(tokenized_inputs["input_ids"]))

{'input_ids': [101, 177, 14190, 125, 9791, 22327, 22320, 253, 785, 11236, 106, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [8]:
tokens = ["A", "aula", "de", "SPLN", "é", "muito", "interessante"]
tokenized_inputs = tokenizer(tokens, is_split_into_words=True)

print(tokenized_inputs)

{'input_ids': [101, 177, 14190, 125, 9791, 22327, 22320, 253, 785, 11236, 102], 'token_type_ids': [0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1]}


In [9]:
print(len(tokens), len(tokenized_inputs["input_ids"]))

7 11


In [10]:
print(tokenizer.convert_ids_to_tokens(tokenized_inputs["input_ids"]))
tokens = ["A", "aula", "de", "SPLN", "é", "muito", "interessante"]

['[CLS]', 'A', 'aula', 'de', 'SP', '##L', '##N', 'é', 'muito', 'interessante', '[SEP]']


In [11]:
print(tokenized_inputs.word_ids())

[None, 0, 1, 2, 3, 3, 3, 4, 5, 6, None]


In [12]:
def align_labels_with_tokens(labels, word_ids):
    new_labels = []
    previous_id = None
    for word_id in word_ids:
        if word_id == None:
            new_labels.append(-100)
        elif previous_id != word_id:
            new_labels.append(labels[word_id])
        else:
            new_labels.append(-100)
        previous_id = word_id

    return new_labels

tokens = ["A", "aula", "de", "SPLN", "é", "muito", "interessante"]
tokenized_inputs = tokenizer(tokens, is_split_into_words=True)
labels = [0,0,0,1,0,0,0]

print(align_labels_with_tokens(labels,tokenized_inputs.word_ids()))

[-100, 0, 0, 0, 1, -100, -100, 0, 0, 0, -100]


In [13]:
def tokenize_dataset(dataset):
    new_dataset = []
    for data in dataset:
        tokenized_inputs = tokenizer(data["tokens"], is_split_into_words=True, truncation=True, max_length=512)
        new_labels = align_labels_with_tokens(data["ner_tags"], tokenized_inputs.word_ids())
        tokenized_inputs["labels"] = new_labels

        new_dataset.append(tokenized_inputs)
    return new_dataset

data_train = tokenize_dataset(raw_datasets["train"])
data_test = tokenize_dataset(raw_datasets["test"])

In [14]:
from datasets import Dataset
train_dataset = Dataset.from_list(data_train)
test_dataset = Dataset.from_list(data_test)
print(train_dataset)
print(test_dataset)

Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 3716
})
Dataset({
    features: ['input_ids', 'token_type_ids', 'attention_mask', 'labels'],
    num_rows: 930
})


In [15]:
from transformers import AutoModelForTokenClassification, TrainingArguments, Trainer

label2id = {l : i for i, l in enumerate(label_list)}
id2label = {v: k for k, v in label2id.items()}
model = AutoModelForTokenClassification.from_pretrained(checkpoint, id2label=id2label , label2id = label2id )

pytorch_model.bin:   0%|          | 0.00/438M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/438M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

BertForTokenClassification LOAD REPORT from: neuralmind/bert-base-portuguese-cased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.seq_relationship.weight                | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
bert.pooler.dense.weight                   | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
cls.predictions.decoder.weight             | UNEXPECTED | 
cls.predictions.bias                       | UNEXPECTED | 
bert.pooler.dense.bias                     | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
classifier.weight                          | MISSING    | 
classifier.bias                            | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok 

In [16]:
batch_size = 8
output_model_name = "my_model"
args = TrainingArguments(
    output_model_name,
    report_to="none",
    eval_strategy = "epoch",
    learning_rate=2e-5,
    per_device_train_batch_size=batch_size, #16
    per_device_eval_batch_size=batch_size, #16
    num_train_epochs=2,
    weight_decay=0.01,
    save_strategy="epoch",
    load_best_model_at_end=True,      # Load the best model after training
    metric_for_best_model="f1",       # Or any metric you use for evaluation
    greater_is_better=True,           # True if higher F1/Accuracy is better
    #save_total_limit=1
    #push_to_hub=True,
)

In [17]:
from transformers import DataCollatorForTokenClassification

data_collator = DataCollatorForTokenClassification(tokenizer)

In [19]:
!pip install evaluate seqeval

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.6/43.6 kB 1.4 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 84.1/84.1 kB 3.1 MB/s eta 0:00:00
  Created wheel for seqeval: filename=seqeval-1.2.2-py3-none-any.whl size=16162 sha256=b29c47b256a250300151e807f5ccf450818dcda8c2e03ac2e7d814c1129101f7
  Stored in directory: /root/.cache/pip/wheels/5f/b8/73/0b2c1a76b701a677653dd79ece07cfabd7457989dbfbdcd8d7
Successfully built seqeval


In [20]:
import numpy as np
import evaluate
seqeval = evaluate.load("seqeval")

def compute_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)

    true_predictions = [
        [label_list[p] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_list[l] for (p, l) in zip(prediction, label) if l != -100]
        for prediction, label in zip(predictions, labels)
    ]

    results = seqeval.compute(predictions=true_predictions, references=true_labels)
    return {
        "precision": results["overall_precision"],
        "recall": results["overall_recall"],
        "f1": results["overall_f1"],
        "accuracy": results["overall_accuracy"],
    }

In [21]:
trainer = Trainer(
    model=model,
    args=args,
    train_dataset=train_dataset,
    eval_dataset=test_dataset,
    processing_class=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

In [22]:
trainer.train()

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.067847,0.940539,0.961380,0.950845,0.983055
2,0.157284,0.062678,0.945578,0.968028,0.956671,0.984762


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['bert.embeddings.LayerNorm.weight', 'bert.embeddings.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.output.LayerNorm.weight', 'bert.encoder.layer.0.output.LayerNorm.bias', 'bert.encoder.layer.1.attention.output.LayerNorm.weight', 'bert.encoder.layer.1.attention.output.LayerNorm.bias', 'bert.encoder.layer.1.output.LayerNorm.weight', 'bert.encoder.layer.1.output.LayerNorm.bias', 'bert.encoder.layer.2.attention.output.LayerNorm.weight', 'bert.encoder.layer.2.attention.output.LayerNorm.bias', 'bert.encoder.layer.2.output.LayerNorm.weight', 'bert.encoder.layer.2.output.LayerNorm.bias', 'bert.encoder.layer.3.attention.output.LayerNorm.weight', 'bert.encoder.layer.3.attention.output.LayerNorm.bias', 'bert.encoder.layer.3.output.LayerNorm.weight', 'bert.encoder.layer.3.output.LayerNorm.bias', 'bert.encoder.layer.4.attention.output.La

Epoch,Training Loss,Validation Loss,Precision,Recall,F1,Accuracy
1,No log,0.067847,0.940539,0.961380,0.950845,0.983055
2,0.157284,0.062678,0.945578,0.968028,0.956671,0.984762


TrainOutput(global_step=930, training_loss=0.10290655525781775, metrics={'train_runtime': 7589.86, 'train_samples_per_second': 0.979, 'train_steps_per_second': 0.123, 'total_flos': 334427858237976.0, 'train_loss': 0.10290655525781775, 'epoch': 2.0})

In [23]:
from transformers import pipeline

classifier = pipeline("ner", model="./my_model/checkpoint-930", aggregation_strategy="first")

# Run inference
text = """O João Paulo, médico, vive no Porto (Portugal) desde 2024.
A Soraia Marques obeteve o seu mestrado na Universidade do Minho em 26 de Maio de 2021.
Depois de obter o grau, foi trabalhar para o o tribunal de Braga como juíza."
"""
classifier(text)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[{'entity_group': 'Pessoa',
  'score': np.float32(0.994719),
  'word': 'João Paulo',
  'start': 2,
  'end': 12},
 {'entity_group': 'Local',
  'score': np.float32(0.9806161),
  'word': 'Porto',
  'start': 30,
  'end': 35},
 {'entity_group': 'Local',
  'score': np.float32(0.96840525),
  'word': 'Portugal',
  'start': 37,
  'end': 45},
 {'entity_group': 'Data',
  'score': np.float32(0.962508),
  'word': '2024',
  'start': 53,
  'end': 57},
 {'entity_group': 'Pessoa',
  'score': np.float32(0.9943749),
  'word': 'Soraia Marques',
  'start': 61,
  'end': 75},
 {'entity_group': 'Organizacao',
  'score': np.float32(0.9126038),
  'word': 'Universidade do Minho',
  'start': 102,
  'end': 123},
 {'entity_group': 'Data',
  'score': np.float32(0.985984),
  'word': '26 de Maio de 2021',
  'start': 127,
  'end': 145},
 {'entity_group': 'Local',
  'score': np.float32(0.94989645),
  'word': 'Braga',
  'start': 206,
  'end': 211},
 {'entity_group': 'Profissao',
  'score': np.float32(0.62998396),
  'word